# MultiModel Testing with AutoGluon

## 1. Setup and Dependencies

In [1]:
import os
import pandas as pd
import numpy as np
import random
from tqdm.auto import tqdm
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import sys
import shutil
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')


C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\.venv1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

In [2]:
logging.info("--- Setting up configuration ---")

if os.path.basename(os.getcwd()) == 'notebooks':
    project_root = os.path.abspath('../..')
else:
    project_root = os.getcwd()

if project_root not in sys.path:
    sys.path.append(project_root)

BASE_DIR = project_root
DATA_DIR = os.path.join(BASE_DIR, "data")
TRAIN_DIR = os.path.join(DATA_DIR, "train_trading_only")
VAL_DIR = os.path.join(DATA_DIR, "val_trading_only")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODELS_DIR = os.path.join(BASE_DIR, "models")

# Define separate output paths for the two models we will train
AUTOGLUON_FINAL_DIR = os.path.join(MODELS_DIR, "autogluon_quick_test")
AUTOGLUON_NO_COVARIATES_DIR = os.path.join(MODELS_DIR, "autogluon_no_covariates")
AUTOGLUON_WITH_COVARIATES_DIR = os.path.join(MODELS_DIR, "autogluon_with_covariates")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(AUTOGLUON_NO_COVARIATES_DIR, exist_ok=True)
os.makedirs(AUTOGLUON_WITH_COVARIATES_DIR, exist_ok=True)

# Model configuration
TARGET_COLS = ["high", "low", "close", "volume"]
OUTPUT_CHUNK_LENGTH = 10
SEED = 827
TIME_LIMIT = 3600
MAX_ASSETS = 20
LIMIT_ROWS_PER_ASSET = None

[INFO] --- Setting up configuration ---


## 3. Load and Convert Data to AutoGluon Format

AutoGluon expects a DataFrame with columns: `[timestamp, item_id, target]`

In [3]:
def load_parquet_files(directory, max_assets=None):
    """Load parquet files from a directory, with an option to limit the count."""
    all_files = [f for f in os.listdir(directory) if f.endswith('.parquet')]
    random.seed(SEED)
    random.shuffle(all_files)
    files = all_files[:max_assets] if max_assets else all_files

    data = {}
    for file in tqdm(files, desc=f"Loading {len(files)} parquet files from {os.path.basename(directory)}"):
        asset_name = file.replace('.parquet', '')
        df = pd.read_parquet(os.path.join(directory, file))
        data[asset_name] = df
    return data

def convert_to_autogluon_format(data_dict, use_covariates=False):
    """
    Convert dict of asset DataFrames to AutoGluon's TimeSeriesDataFrame format.
    If use_covariates is True, all non-target columns are preserved.
    """
    all_data = []

    for asset_name, df in tqdm(data_dict.items(), desc=f"Converting data (covariates: {use_covariates})"):
        df = df.copy()
        if LIMIT_ROWS_PER_ASSET:
            df = df.iloc[-LIMIT_ROWS_PER_ASSET:]

        df['ExecutionTime'] = pd.to_datetime(df['ExecutionTime'])
        if df['ExecutionTime'].dt.tz is not None:
            df['ExecutionTime'] = df['ExecutionTime'].dt.tz_localize(None)

        df.rename(columns={"ExecutionTime": "timestamp"}, inplace=True)

        if use_covariates:
            # Melt the dataframe to create a long format, keeping covariates
            id_vars = [c for c in df.columns if c not in TARGET_COLS]
            melted_df = df.melt(
                id_vars=id_vars,
                value_vars=TARGET_COLS,
                var_name="target_col_name",
                value_name="target"
            )
            melted_df["item_id"] = asset_name + "_" + melted_df["target_col_name"]
            melted_df.drop(columns=["target_col_name"], inplace=True)
            all_data.append(melted_df)
        else:
            # Original logic: create separate series for each target, no covariates
            for col in TARGET_COLS:
                if col in df.columns:
                    item_df = pd.DataFrame({
                        'timestamp': df['timestamp'].values,
                        'item_id': f"{asset_name}_{col}",
                        'target': df[col].astype(np.float32).values
                    })
                    all_data.append(item_df)

    combined_df = pd.concat(all_data, ignore_index=True)
    ts_df = TimeSeriesDataFrame.from_data_frame(
        combined_df,
        id_column='item_id',
        timestamp_column='timestamp'
    )
    return ts_df

## 4. Compare Global Models with AutoGluon

We'll use multiple hyperparameter configurations for Deep Learning models to simulate:
Zero-Shot (minimal train) vs Fine-Tuned.

In [4]:
def train_autogluon_model(train_data, val_data, predictor_path, known_covariates_names=None):
    """A helper function to train an AutoGluon TimeSeriesPredictor."""
    if known_covariates_names is None:
        known_covariates_names = []

    hyperparameters = {
        "DeepAR": [
            {"max_epochs": 1, "ag_args": {"name_suffix": "_ZeroShotBase"}},
            {"max_epochs": 500, "ag_args": {"name_suffix": "_FineTuned"}},
        ],
        "TemporalFusionTransformer": [
            {"max_epochs": 1, "ag_args": {"name_suffix": "_ZeroShotBase"}},
            {"max_epochs": 500, "ag_args": {"name_suffix": "_FineTuned"}},
        ],
        "PatchTST": [
            {"max_epochs": 1, "ag_args": {"name_suffix": "_ZeroShotBase"}},
            {"max_epochs": 500, "ag_args": {"name_suffix": "_FineTuned"}},
        ]
    }

    predictor = TimeSeriesPredictor(
        prediction_length=OUTPUT_CHUNK_LENGTH,
        path=predictor_path,
        target="target",
        eval_metric="sMAPE",
        freq="15min",
        known_covariates_names=known_covariates_names,
        verbosity=2,
    )

    predictor.fit(
        train_data=train_data,
        hyperparameters=hyperparameters,
        time_limit=TIME_LIMIT,
        random_seed=SEED
    )

    logging.info(f"--- Evaluating model from {predictor_path} ---")
    leaderboard = predictor.leaderboard(val_data, silent=True)
    best_score = leaderboard.iloc[0]["score_test"]

    return predictor, leaderboard, best_score


## 5. Evaluate Models

In [5]:
logging.info("--- Loading raw data ---")
raw_train_data = load_parquet_files(TRAIN_DIR, max_assets=MAX_ASSETS)
raw_val_data = load_parquet_files(VAL_DIR, max_assets=MAX_ASSETS)

[INFO] --- Loading raw data ---
Loading 20 parquet files from val_trading_only: 100%|███████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 420.55it/s]


### 5.1 Training with no covariates

In [6]:
logging.info("--- Starting Run 1: Training WITHOUT covariates ---")
train_ts_no_cov = convert_to_autogluon_format(raw_train_data, use_covariates=False)
val_ts_no_cov = convert_to_autogluon_format(raw_val_data, use_covariates=False)

_, leaderboard_no_cov, score_no_cov = train_autogluon_model(
    train_data=train_ts_no_cov,
    val_data=val_ts_no_cov,
    predictor_path=AUTOGLUON_NO_COVARIATES_DIR
)
logging.info(f"Finished Run 1 (No Covariates). Best validation sMAPE: {score_no_cov:.4f}")
print(leaderboard_no_cov)

[INFO] --- Starting Run 1: Training WITHOUT covariates ---
Converting data (covariates: False): 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 199.97it/s]
Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to 'C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\autogluon_no_covariates'
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.11
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          32
GPU Count:          1
Memory Avail:       15.31 GB / 31.63 GB (48.4%)
Disk Space Avail:   290.71 GB / 928.35 GB (31.3%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': SMAPE,
 'freq': '15min',
 'hyperparameters': {'DeepAR': [{'ag_args': {'name_suffix': '_ZeroShotBase'},
                                 'max_epochs': 1},
                               

                                    model  score_test  score_val  \
0  TemporalFusionTransformer_ZeroShotBase   -1.560565  -1.802817   
1     TemporalFusionTransformer_FineTuned   -1.588635  -1.741027   
2                        WeightedEnsemble   -1.588947  -1.741026   
3                        DeepAR_FineTuned   -1.607222  -1.809059   
4                      PatchTST_FineTuned   -1.655960  -1.769055   
5                   PatchTST_ZeroShotBase   -1.696378  -1.749923   
6                     DeepAR_ZeroShotBase   -1.737299  -1.817098   

   pred_time_test  pred_time_val  fit_time_marginal  fit_order  
0        0.688276       1.109955           8.129805          1  
1        0.686326       1.235458         240.327399          2  
2        0.868889       1.405412           1.116786          7  
3        0.792911       1.449392          79.893280          4  
4        0.160520       0.166634          79.990810          6  
5        0.181563       0.169954           2.175226          5  


### 5.2 Training with Covariates

In [7]:
logging.info("--- Starting Run 2: Training WITH covariates ---")
train_ts_with_cov = convert_to_autogluon_format(raw_train_data, use_covariates=True)
val_ts_with_cov = convert_to_autogluon_format(raw_val_data, use_covariates=True)

# Get covariate names from the dataframe, excluding metadata columns
known_covariates = [
    col for col in train_ts_with_cov.columns
    if col not in ["target", "item_id", "timestamp"]
]
logging.info(f"Found {len(known_covariates)} covariates to use.")

_, leaderboard_with_cov, score_with_cov = train_autogluon_model(
    train_data=train_ts_with_cov,
    val_data=val_ts_with_cov,
    predictor_path=AUTOGLUON_WITH_COVARIATES_DIR,
    known_covariates_names=known_covariates
)
logging.info(f"Finished Run 2 (With Covariates). Best validation sMAPE: {score_with_cov:.4f}")
print(leaderboard_with_cov)

[INFO] --- Starting Run 2: Training WITH covariates ---
Converting data (covariates: True): 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:01<00:00, 10.38it/s]
[INFO] Found 27 covariates to use.
Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to 'C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\autogluon_with_covariates'
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.11
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          32
GPU Count:          1
Memory Avail:       10.05 GB / 31.63 GB (31.8%)
Disk Space Avail:   290.56 GB / 928.35 GB (31.3%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': SMAPE,
 'freq': '15min',
 'hyperparameters': {'DeepAR': [{'ag_args': {'name_suffix': '_ZeroShotBase'},
                                 'max_epochs': 1

                                    model  score_test  score_val  \
0     TemporalFusionTransformer_FineTuned   -1.598400  -1.786069   
1                        DeepAR_FineTuned   -1.727938  -1.999984   
2                     DeepAR_ZeroShotBase   -1.788398  -1.838337   
3                   PatchTST_ZeroShotBase   -1.878066  -1.896435   
4                        WeightedEnsemble   -1.889324  -1.727613   
5  TemporalFusionTransformer_ZeroShotBase   -1.972135  -1.752598   
6                      PatchTST_FineTuned   -1.999835  -2.000000   

   pred_time_test  pred_time_val  fit_time_marginal  fit_order  
0        4.008917       8.480537         397.367243          2  
1        4.392831       8.680718         674.794756          4  
2        4.324722       8.861549          28.217400          3  
3        3.507390      30.024212          30.487523          5  
4       16.727478      33.459488           1.558227          7  
5        4.523196       9.027311          47.323625          1  


## 6. Save Final Results

In [8]:
# --- Compare models and save the best one ---
logging.info("--- Comparing models and saving the best one ---")

# Since sMAPE is negative (higher is better), we find the max score
if score_with_cov >= score_no_cov:
    logging.info(f"Model WITH covariates is better ({score_with_cov:.4f} vs {score_no_cov:.4f}).")
    best_model_path = AUTOGLUON_WITH_COVARIATES_DIR
else:
    logging.info(f"Model WITHOUT covariates is better ({score_no_cov:.4f} vs {score_with_cov:.4f}).")
    best_model_path = AUTOGLUON_NO_COVARIATES_DIR

logging.info(f"Copying best model from {best_model_path} to final destination {AUTOGLUON_FINAL_DIR}")
if os.path.exists(AUTOGLUON_FINAL_DIR):
    shutil.rmtree(AUTOGLUON_FINAL_DIR)
shutil.copytree(best_model_path, AUTOGLUON_FINAL_DIR)

logging.info("--- Script finished successfully! ---")

[INFO] --- Comparing models and saving the best one ---
[INFO] Model WITHOUT covariates is better (-1.5606 vs -1.5984).
[INFO] Copying best model from C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\autogluon_no_covariates to final destination C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\autogluon_quick_test
[INFO] --- Script finished successfully! ---
